# Worked example: primary issuance is not liquidity

This notebook demonstrates the one thing `rwa-liquidity` gets right that a
naive implementation gets wrong. It runs on the committed sample dataset, so
it needs no API keys.

> **The data here is synthetic.** It was constructed to demonstrate the
> metrics, not observed. See `data/sample/README.md`.

## 1. Load the sample data

The sample passes through the same `validate` boundary as any adapter's
output, so this exercises the real ingestion path rather than a shortcut
around it.

In [1]:
from rwa_liquidity.demo import load_demo_dataset

data = load_demo_dataset()
print(f"window     {data.window}")
print(f"snapshots  {data.snapshots.height}")
print(f"transfers  {data.transfers.height}")
print(f"holders    {data.holders.height}")

window     [2026-06-01T00:00:00+00:00, 2026-07-01T00:00:00+00:00)
snapshots  4
transfers  23
holders    26


## 2. What the transfers actually are

Every one of these emitted the same on-chain `Transfer` event. The `kind`
column is what separates issuance from trading, and it is assigned by the
adapter, which is the only layer that knows an asset's issuer addresses.

In [2]:
import polars as pl

(
    data.transfers.join(
        data.snapshots.select("asset_uid", "symbol").unique(),
        on="asset_uid",
    )
    .group_by("symbol", "kind")
    .agg(pl.len().alias("transfers"), pl.col("amount").sum().alias("volume"))
    .sort("symbol", "kind")
)

symbol,kind,transfers,volume
str,str,u32,f64
"""SYNTH-CREDIT""","""mint""",1,20000.0
"""SYNTH-CREDIT""","""secondary""",2,800.0
"""SYNTH-CREDIT""","""unclassified""",1,1200.0
"""SYNTH-GOLD""","""burn""",1,10000.0
"""SYNTH-GOLD""","""mint""",2,80000.0
"""SYNTH-GOLD""","""secondary""",10,250000.0
"""SYNTH-TBILL""","""burn""",2,5e7
"""SYNTH-TBILL""","""mint""",4,2.8e8


`SYNTH-TBILL` has four mints and two burns, and **not a single secondary
transfer**. Its investors only ever traded with the issuer.

## 3. The same asset, measured two ways

`turnover_ratio` takes a `mode`. Nothing else changes: same data, same
window, same formula.

In [3]:
from rwa_liquidity.metrics import turnover_ratio
from rwa_liquidity.schema.types import VolumeMode

TBILL = "ethereum:0x0000000000000000000000000000000000000001"
transfers = data.transfers.filter(pl.col("asset_uid") == TBILL)
snapshots = data.snapshots.filter(pl.col("asset_uid") == TBILL)

for mode in (VolumeMode.ALL, VolumeMode.SECONDARY_ONLY):
    result = turnover_ratio(transfers, snapshots, window=data.window, mode=mode)
    print(f"{mode:>16}  turnover = {result.value:.4f}")

             all  turnover = 0.6600
  secondary_only  turnover = 0.0000


An implementation that counts issuance as trading reports a fund with
two-thirds of its supply turning over in a month. The honest reading is that
it has no secondary market at all.

Note that `0.0000` here is a real measurement, not a missing value. The
package returns `None` when a metric is undefined and never conflates the
two.

## 4. Every number carries its provenance

A value alone is not a result. If this figure ends up in a paper, this is
what justifies it.

In [4]:
result = turnover_ratio(transfers, snapshots, window=data.window)
p = result.provenance

print(f"value        {result.value}")
print(f"metric       {p.metric}")
print(f"asset        {p.asset_uid}")
print(f"sources      {p.sources}")
print(f"window       {p.window}")
print(f"mode         {p.mode}")
print(f"denomination {p.denomination}")
print(f"records      {p.n_records}")
for note in p.exclusions:
    print(f"excluded     {note}")

value        0.0
metric       turnover_ratio
asset        ethereum:0x0000000000000000000000000000000000000001
sources      ('sample_chain', 'sample_protocol_feed', 'sample_registry')
window       [2026-06-01T00:00:00+00:00, 2026-07-01T00:00:00+00:00)
mode         secondary_only
denomination native
records      0
excluded     6 of 6 in-window transfers excluded by mode 'secondary_only'


## 5. Dormancy tells the same story

The volume mode applies to the participation metrics too. An address that
received a mint and never traded is dormant: taking delivery of an issuance
is not market participation.

In [5]:
from rwa_liquidity.metrics import dormancy

holders = data.holders.filter(pl.col("asset_uid") == TBILL)
for mode in (VolumeMode.ALL, VolumeMode.SECONDARY_ONLY):
    result = dormancy(holders, transfers, snapshots, window=data.window, mode=mode)
    print(f"{mode:>16}  dormancy = {result.value:.1%}")

             all  dormancy = 3.0%
  secondary_only  dormancy = 100.0%


## 6. The full table, and its caveats

`build_report` computes every metric for every asset. Warnings are carried
forward rather than summarised away: six numbers that all look equally solid
are less useful than six numbers with a note saying which are provisional.

In [6]:
from rwa_liquidity.metrics.report import build_report, report_frame

reports = build_report(data.snapshots, data.transfers, data.holders, window=data.window)
report_frame(reports).select(
    "symbol",
    "turnover_ratio",
    "active_holder_ratio",
    "top_10_holder_share",
    "holder_hhi",
    "dormancy",
    "n_warnings",
)

symbol,turnover_ratio,active_holder_ratio,top_10_holder_share,holder_hhi,dormancy,n_warnings
str,f64,f64,f64,f64,f64,i64
"""SYNTH-TBILL""",0.0,0.0,1.0,2586.0,1.0,2
"""SYNTH-GOLD""",0.25,0.833333,0.95,1086.5,0.05,0
"""SYNTH-CREDIT""",0.008,0.075,0.98,3586.0,0.13,3


In [7]:
for report in reports:
    for warning in report.warnings:
        print(f"{report.symbol}:\n  {warning}\n")

SYNTH-TBILL:
  no addresses were active in the window, so the ratio is undefined

SYNTH-TBILL:
  only 8 holders are known, fewer than the 10 requested; the share covers all of them

SYNTH-CREDIT:
  1 of 4 transfers (25.0%) could not be classified as primary or secondary and are excluded under mode 'secondary_only'; the value is a lower bound on activity

SYNTH-CREDIT:
  the holder list covers 6 of 40 reported holders (15.0%); a truncated distribution biases concentration downward, so the true value is higher than reported

SYNTH-CREDIT:
  only 6 holders are known, fewer than the 10 requested; the share covers all of them



`SYNTH-CREDIT` is the instructive one. Its holder list covers 6 of 40
reported holders, so its concentration figures are biased downward and the
true values are higher than shown. That cannot be corrected, only disclosed.

## 7. Where sources disagree

Two sources describe `SYNTH-TBILL` and report different supplies. The
package reports the gap and picks neither: which source is right is a
research question about the providers, not something a library can decide.

In [8]:
from rwa_liquidity.reconcile import reconcile_snapshots

report = reconcile_snapshots(data.snapshots)
print(f"{report.compared} comparisons, {len(report.disagreements)} disagreements\n")
for item in report.disagreements:
    print(item.describe())

3 comparisons, 2 disagreements



ethereum:0x0000000000000000000000000000000000000001 total_supply: sample_protocol_feed=560,000,000 vs sample_registry=500,000,000 (10.7% apart)
ethereum:0x0000000000000000000000000000000000000001 market_value_usd: sample_protocol_feed=560,000,000 vs sample_registry=500,000,000 (10.7% apart)


This mirrors a real case: DeFiLlama's protocol TVL for BlackRock's BUIDL
covers two share classes, so it legitimately exceeds the value of the single
contract rwa.xyz reports on. Neither source is wrong. The disagreement is
informative, and burying it by picking a winner would lose that.

---

## Further reading

- [`docs/methodology.md`](../docs/methodology.md) — every formula and caveat
- [`docs/data-sources.md`](../docs/data-sources.md) — what each provider
  supplies and what is wrong with it
- [`DECISIONS.md`](../DECISIONS.md) — why the design is the way it is